In [39]:
from datasets import load_dataset
import ujson as json, re

dataset_lcb = list(load_dataset("livecodebench/code_generation_lite", version_tag="release_v6", trust_remote_code=True)["test"])
for i, d in enumerate(dataset_lcb):
    d["task_id"] = f"livecodebench_codegen_{i}"
    d["sample_date"] = d["contest_date"][:10]
    d["source"] = "livecodebench"
    d["task"] = "code"
    d["split"] = "test"

    d["metadata"] = json.loads(d["metadata"])
    # if "func_name" not in d["metadata"] and "def " in d["starter_code"]:
    #     starter_code_lines = d["starter_code"].split("\n")
    #     # 'class Solution:\n    def countSeniors(self, details: List[str]) -> int:\n        
    #     print("------------")
    #     print(d["starter_code"])
    #     d["metadata"]["func_name"] = starter_code_lines[1].split("def ")[1].split("(")[0]
    #     print(d["metadata"]["func_name"])

print(Counter([k for d in dataset_lcb for k in d]))
# dataset_lcb_problem = [d for d in dataset_lcb if "func_name" not in d["metadata"]]
dataset_lcb = [d for d in dataset_lcb if "func_name" in d["metadata"]]

print(len(dataset_lcb))

Counter({'question_title': 1055, 'question_content': 1055, 'platform': 1055, 'question_id': 1055, 'contest_id': 1055, 'contest_date': 1055, 'starter_code': 1055, 'difficulty': 1055, 'public_test_cases': 1055, 'private_test_cases': 1055, 'metadata': 1055, 'task_id': 1055, 'sample_date': 1055, 'source': 1055, 'task': 1055, 'split': 1055})
444


In [40]:
dataset_he = list(load_dataset("openai/openai_humaneval", trust_remote_code=True)["test"])

def extract_test_cases(test):
    test_cases = []
    for line in test.split("\n"):
        line = line.strip()
        if line.startswith("assert"):
            try:
                line_inp, line_out = line.split("==")
                # use re to find what's inside the parentheses
                inp = re.findall(r'\[(.*?)\]', line_inp)
                out = line_out.strip()
                test_cases.append({"input": inp, "output": out, "testtype": "functional"})
            except:
                # print(line)
                pass
    return json.dumps(test_cases)

for d in dataset_he:
    d["public_test_cases"] = extract_test_cases(d["test"])
    d["sample_date"] = "2021-07-07"
    d["metadata"] = {"func_name": d["entry_point"]}
    d["source"] = "openai_humaneval"
    d["task"] = "code"
    d["split"] = "test"


In [41]:
from collections import Counter

final_dataset = dataset_lcb + dataset_he

for d in final_dataset:
    d["shards"] = []

print(Counter([d["source"] for d in final_dataset]))

with open("data/all_code_samples.json", "w") as f:
    json.dump(final_dataset, f)


Counter({'livecodebench': 444, 'openai_humaneval': 164})
